# Notebook 105 — Refinamiento sistemático

Tienes un agente parametrizado y sabes medirlo. Falta la parte que separa el trabajo serio
del ensayo y error: **elegir una configuración de forma defendible.**

Vas a comparar cinco configuraciones bajo un principio estricto: **una sola variable cambia a
la vez**, y todas se miden sobre exactamente la misma muestra. Si cambias `k` y el prompt al
mismo tiempo y el resultado mejora, no sabes a qué atribuirlo.

## El protocolo

1. **Barrido en `dev`** — cinco configuraciones, misma muestra, misma semilla
2. **Selección** — la mejor por `citation_f1`
3. **Validación única en `holdout`** — datos que no se usaron para elegir

El paso 3 es el que casi nadie hace y el que más importa. Si eliges la mejor de cinco
configuraciones midiendo sobre la misma muestra, una parte de esa ventaja es ruido que
aprovechaste sin querer. El holdout te dice cuánta.

In [ ]:
%run ./103_rag_agent

In [ ]:
dbutils.widgets.text("n_eval", "20", "Preguntas por configuración")
N_EVAL = int(dbutils.widgets.get("n_eval"))

print(f"Preguntas por configuración : {N_EVAL}")
print(f"Configuraciones a probar    : 5")
print(f"Llamadas al LLM estimadas   : ~{N_EVAL * 5}")

## 1. Scorers

Los mismos del notebook 104. Los redefinimos aquí para que este notebook sea autónomo: puedes
ejecutarlo sin haber corrido el 104 en esta sesión.

In [ ]:
from mlflow.genai.scorers import scorer
from mlflow.entities import Feedback
import re as _re


def _prf(cited, gold) -> tuple[float, float, float]:
    c, g = set(cited or []), set(gold or [])
    if not g:
        return 0.0, 0.0, 0.0
    inter = len(c & g)
    p = inter / len(c) if c else 0.0
    r = inter / len(g)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1


@scorer
def citation_precision(outputs, expectations) -> float:
    return _prf((outputs or {}).get("cited_chunk_ids", []),
                (expectations or {}).get("gold_chunk_ids", []))[0]


@scorer
def citation_recall(outputs, expectations) -> float:
    return _prf((outputs or {}).get("cited_chunk_ids", []),
                (expectations or {}).get("gold_chunk_ids", []))[1]


@scorer
def citation_f1(outputs, expectations) -> float:
    return _prf((outputs or {}).get("cited_chunk_ids", []),
                (expectations or {}).get("gold_chunk_ids", []))[2]


@scorer
def retrieval_hit(outputs, expectations) -> float:
    retrieved = set((outputs or {}).get("retrieved_chunk_ids", []))
    gold = set((expectations or {}).get("gold_chunk_ids", []))
    return 1.0 if retrieved & gold else 0.0


@scorer
def answer_token_f1(outputs, expectations) -> float:
    def norm(t):
        return set(_re.findall(r"\w+", (t or "").lower()))
    pred = norm((outputs or {}).get("answer", ""))
    gold = norm((expectations or {}).get("expected_response", ""))
    if not pred or not gold:
        return 0.0
    inter = len(pred & gold)
    if inter == 0:
        return 0.0
    p, r = inter / len(pred), inter / len(gold)
    return 2 * p * r / (p + r)


SCORERS = [citation_precision, citation_recall, citation_f1, retrieval_hit, answer_token_f1]

## 2. Muestra fija

`orderBy("question_id").limit(N)` es determinístico: devuelve las mismas preguntas en cada
corrida y en cada configuración. Sin esto, estarías comparando configuraciones sobre datos
distintos y la comparación no valdría nada.

In [ ]:
dev_sample = spark.table(T_DEV).orderBy("question_id").limit(N_EVAL).collect()

eval_data = [
    {
        "inputs": {"question": r["question"]},
        "expectations": {
            "expected_response": r["answer"],
            "gold_chunk_ids": list(r["gold_chunk_ids"]),
        },
    }
    for r in dev_sample
]

print(f"Muestra dev fijada: {len(eval_data)} preguntas")
print(f"Primera: {eval_data[0]['inputs']['question'][:70]}...")

## 3. Las configuraciones

Cinco combinaciones, cada una diseñada para responder una pregunta concreta:

| # | Config | Pregunta que responde |
|---|---|---|
| 1 | k=3, v1, model | Línea base |
| 2 | k=5, v1, model | ¿Recuperar más chunks ayuda? |
| 3 | k=3, v2, intersect | ¿El prompt de "conjunto mínimo" mejora la precisión? |
| 4 | k=10, v2, intersect | ¿Recuperar amplio + citar estrecho es lo mejor de ambos mundos? |
| 5 | k=5, v1, all_retrieved | Control: la política ingenua, para cuantificar cuánto cuesta |

La configuración 5 no está para ganar. Está para que veas con números el costo de citar sin
criterio.

In [ ]:
CONFIGS = [
    {"k": 3, "prompt_version": "v1", "citation_policy": "model"},
    {"k": 5, "prompt_version": "v1", "citation_policy": "model"},
    {"k": 3, "prompt_version": "v2", "citation_policy": "intersect"},
    {"k": 10, "prompt_version": "v2", "citation_policy": "intersect"},
    {"k": 5, "prompt_version": "v1", "citation_policy": "all_retrieved"},
]

for i, c in enumerate(CONFIGS, 1):
    print(f"  {i}. k={c['k']:<3} prompt={c['prompt_version']}  política={c['citation_policy']}")

## 4. Ejecutar el barrido

Cada configuración se registra como un run independiente, con sus parámetros. Así la UI de
MLflow puede compararlos después.

In [ ]:
import mlflow
import pandas as pd
import time

sweep_rows = []

for i, cfg in enumerate(CONFIGS, 1):
    run_name = f"sweep_k{cfg['k']}_{cfg['prompt_version']}_{cfg['citation_policy']}"
    print(f"\n[{i}/{len(CONFIGS)}] {run_name}")

    def predict_fn(question: str, _c=cfg) -> dict:
        return rag_agent(
            question,
            k=_c["k"],
            prompt_version=_c["prompt_version"],
            citation_policy=_c["citation_policy"],
        )

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params({**cfg, "retriever": "ai_search",
                           "llm_endpoint": LLM_ENDPOINT, "n_eval": len(eval_data),
                           "split": "dev"})

        res = mlflow.genai.evaluate(data=eval_data, predict_fn=predict_fn, scorers=SCORERS)

    row = {"config_id": i, **cfg, "run_id": run.info.run_id}
    row.update({k.replace("/mean", ""): round(v, 4)
                for k, v in res.metrics.items() if "/mean" in k})
    sweep_rows.append(row)

    print(f"      f1={row.get('citation_f1', 0):.3f}  "
          f"prec={row.get('citation_precision', 0):.3f}  "
          f"rec={row.get('citation_recall', 0):.3f}")
    time.sleep(1)

sweep = pd.DataFrame(sweep_rows)
print("\nBarrido completado.")

## 5. Resultados del barrido

In [ ]:
cols = ["config_id", "k", "prompt_version", "citation_policy",
        "citation_precision", "citation_recall", "citation_f1",
        "retrieval_hit", "answer_token_f1"]
cols = [c for c in cols if c in sweep.columns]

display(sweep[cols].sort_values("citation_f1", ascending=False))

### El trade-off, visto de frente

In [ ]:
print("=" * 74)
print(f"  {'CONFIG':<28} {'PRECISION':>10} {'RECALL':>9} {'F1':>8} {'HIT':>7}")
print("=" * 74)
for _, r in sweep.sort_values("citation_f1", ascending=False).iterrows():
    label = f"k={r['k']} {r['prompt_version']} {r['citation_policy']}"
    print(f"  {label:<28} {r.get('citation_precision', 0):>10.3f} "
          f"{r.get('citation_recall', 0):>9.3f} {r.get('citation_f1', 0):>8.3f} "
          f"{r.get('retrieval_hit', 0):>7.3f}")
print("=" * 74)

naive = sweep[sweep["citation_policy"] == "all_retrieved"]
best_row = sweep.loc[sweep["citation_f1"].idxmax()]
if len(naive):
    n = naive.iloc[0]
    print(f"\n  Política ingenua (all_retrieved):")
    print(f"    recall={n.get('citation_recall', 0):.3f} (alto, como esperabas)")
    print(f"    precision={n.get('citation_precision', 0):.3f} (el costo de citar todo)")
    print(f"    F1={n.get('citation_f1', 0):.3f} vs {best_row['citation_f1']:.3f} de la mejor")

Este contraste es la lección central del notebook. La política `all_retrieved` maximiza el
recall — casi nunca omite evidencia — y aun así pierde, porque diluye cada cita correcta
entre varias incorrectas.

Es la misma tensión que enfrenta cualquier sistema que muestra fuentes a un usuario. Si
adjuntas cinco referencias donde una bastaba, técnicamente "citaste la fuente", pero le
trasladaste al lector el trabajo de encontrar cuál era. La métrica de la competencia
codifica ese juicio de producto.

## 6. Elegir la configuración ganadora

Criterio: máximo `citation_f1`; en caso de empate, mayor `answer_token_f1`. El criterio se
fija **antes** de ver los resultados — si eliges la métrica después de mirar los números,
estás eligiendo la que favorece tu hipótesis preferida.

In [ ]:
ranked = sweep.sort_values(
    ["citation_f1", "answer_token_f1"], ascending=[False, False]
).reset_index(drop=True)

best = ranked.iloc[0]
BEST_CONFIG = {
    "k": int(best["k"]),
    "prompt_version": str(best["prompt_version"]),
    "citation_policy": str(best["citation_policy"]),
}

print("Configuración ganadora en dev:")
for key, val in BEST_CONFIG.items():
    print(f"  {key:<18} {val}")
print(f"\n  citation_f1        {best['citation_f1']:.3f}")
print(f"  citation_precision {best.get('citation_precision', 0):.3f}")
print(f"  citation_recall    {best.get('citation_recall', 0):.3f}")

margen = best["citation_f1"] - ranked.iloc[1]["citation_f1"]
print(f"\n  Margen sobre la segunda: {margen:.3f}")
if margen < 0.05:
    print("  Margen estrecho: con esta muestra, la diferencia podría ser ruido.")

## 7. Validación en holdout

Acabas de elegir entre cinco configuraciones mirando `dev`. Esa elección **usó información**
de esa muestra, así que el F1 que obtuviste ahí es optimista por construcción.

El holdout no participó en ninguna decisión. Su número es tu estimación honesta.

Una sola medición, una sola configuración. Si midieras las cinco en holdout y eligieras la
mejor, habrías convertido el holdout en otro dev y estarías de vuelta en el mismo problema.

In [ ]:
holdout_sample = spark.table(T_HOLDOUT).orderBy("question_id").limit(N_EVAL).collect()

holdout_data = [
    {
        "inputs": {"question": r["question"]},
        "expectations": {
            "expected_response": r["answer"],
            "gold_chunk_ids": list(r["gold_chunk_ids"]),
        },
    }
    for r in holdout_sample
]

print(f"Muestra holdout: {len(holdout_data)} preguntas (nunca usadas para elegir)")

In [ ]:
with mlflow.start_run(
    run_name=f"holdout_k{BEST_CONFIG['k']}_{BEST_CONFIG['prompt_version']}_{BEST_CONFIG['citation_policy']}"
) as run_hold:
    mlflow.log_params({**BEST_CONFIG, "retriever": "ai_search",
                       "llm_endpoint": LLM_ENDPOINT, "n_eval": len(holdout_data),
                       "split": "holdout"})

    res_hold = mlflow.genai.evaluate(
        data=holdout_data,
        predict_fn=lambda question: rag_agent(question, **BEST_CONFIG),
        scorers=SCORERS,
    )

m_hold = {k.replace("/mean", ""): round(v, 4)
          for k, v in res_hold.metrics.items() if "/mean" in k}

In [ ]:
comparativa = pd.DataFrame({
    "dev": {m: best.get(m, float("nan"))
            for m in ["citation_precision", "citation_recall", "citation_f1",
                      "retrieval_hit", "answer_token_f1"]},
    "holdout": {m: m_hold.get(m, float("nan"))
                for m in ["citation_precision", "citation_recall", "citation_f1",
                          "retrieval_hit", "answer_token_f1"]},
})
comparativa["diferencia"] = comparativa["holdout"] - comparativa["dev"]

print("=" * 60)
print("  CONFIGURACIÓN GANADORA: DEV vs HOLDOUT")
print("=" * 60)
display(comparativa.reset_index().rename(columns={"index": "métrica"}))

caida = comparativa.loc["citation_f1", "dev"] - comparativa.loc["citation_f1", "holdout"]
print(f"\n  Caída de citation_f1: {caida:+.3f}")
if caida > 0.15:
    print("  Caída considerable: parte de la ventaja en dev era ruido de la muestra.")
elif caida > 0.05:
    print("  Caída moderada: esperable al elegir entre varias configuraciones.")
else:
    print("  La configuración generaliza bien a datos no vistos.")

## 8. Persistir la configuración ganadora

El notebook 106 lee este archivo. Guardar la decisión en disco (en vez de copiarla a mano)
evita que la submission se genere con una configuración distinta de la que validaste.

In [ ]:
import json

CONFIG_PATH = f"{VOL}/best_config.json"

payload = {
    **BEST_CONFIG,
    "dev_citation_f1": float(best["citation_f1"]),
    "holdout_citation_f1": float(m_hold.get("citation_f1", 0.0)),
    "llm_endpoint": LLM_ENDPOINT,
    "n_eval_dev": len(eval_data),
    "n_eval_holdout": len(holdout_data),
    "selected_at": pd.Timestamp.utcnow().isoformat(),
}

with open(CONFIG_PATH, "w", encoding="utf-8") as fh:
    json.dump(payload, fh, indent=2)

print(f"Guardado en {CONFIG_PATH}:\n")
print(json.dumps(payload, indent=2))

### Guardar también el barrido completo

In [ ]:
T_SWEEP = f"{CATALOG}.{SCHEMA}.agenteval_sweep_results"

sweep_out = sweep.copy()
sweep_out["split"] = "dev"
sweep_out["evaluated_at"] = pd.Timestamp.utcnow().isoformat()

(spark.createDataFrame(sweep_out)
    .write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(T_SWEEP))

display(spark.table(T_SWEEP).select(*[c for c in cols if c in sweep_out.columns]))

## 9. Comparar el barrido en la UI

1. Menú lateral → **Experiments** → `agenteval_rag`
2. Filtra por nombre: escribe `sweep_` en el buscador de runs
3. Selecciona los cinco runs → **Compare**
4. En la vista de comparación, elige el gráfico **Parallel Coordinates**
5. Pon `k` y `citation_policy` como ejes de parámetros, y `citation_f1` como métrica

El gráfico de coordenadas paralelas hace visible de un vistazo qué combinaciones de
parámetros llevan a los F1 altos. Con cinco configuraciones es casi decorativo; con
cincuenta, es la única forma práctica de leer un barrido.